In [0]:
# Azure Databricks Assessment - City Water Network Monitoring
# SETUP ONLY: this notebook creates tables, not exercise solutions.
# Upload the five data files into the Unity Catalog Volume folder before running.

In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
)

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "water_monitoring")
dbutils.widgets.text("volume", "landing")
dbutils.widgets.text("folder", "water_network")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume = dbutils.widgets.get("volume").strip()
folder = dbutils.widgets.get("folder").strip().strip("/")

for label, value in [("catalog", catalog), ("schema", schema), ("volume", volume)]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
        raise ValueError(f"Invalid {label}: use letters, digits and underscores only")
if not re.fullmatch(r"[A-Za-z0-9_/-]+", folder) or ".." in folder:
    raise ValueError("Invalid folder name")

namespace = f"`{catalog}`.`{schema}`"
base_path = f"/Volumes/{catalog}/{schema}/{volume}/{folder}"
print("Data location:", base_path)

required = [
    "water_meter_readings.csv", "water_zones.json", "water_service_requests.csv",
    "water_reading_changes.csv", "water_readings_incremental.csv"
]
try:
    available = {entry.name.rstrip("/") for entry in dbutils.fs.ls(base_path)}
except Exception as exc:
    raise RuntimeError(f"Cannot access {base_path}. Create the Volume and upload the files first.") from exc
missing = sorted(set(required) - available)
if missing:
    raise FileNotFoundError(f"Missing data files in {base_path}: {missing}")

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA water_monitoring")

def save_delta(df, name):
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{catalog}.{schema}.{name}"))

Data location: /Volumes/workspace/water_monitoring/landing/water_network


In [0]:
reading_schema = StructType([
    StructField("reading_id", IntegerType(), True),
    StructField("meter_id", StringType(), True),
    StructField("zone_id", StringType(), True),
    StructField("reading_time", TimestampType(), True),
    StructField("flow_lpm", DoubleType(), True),
    StructField("pressure_bar", DoubleType(), True),
    StructField("chlorine_mg_l", DoubleType(), True),
    StructField("consumption_liters", IntegerType(), True),
    StructField("reading_status", StringType(), True),
    StructField("source_system", StringType(), True),
])

def read_readings(filename):
    return (spark.read.format("csv")
        .option("header", "true")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .schema(reading_schema)
        .load(f"{base_path}/{filename}"))

source = read_readings("water_meter_readings.csv")
save_delta(source, "water_meter_readings_source")
save_delta(source, "water_meter_readings_work")
save_delta(read_readings("water_reading_changes.csv"), "water_reading_changes")
save_delta(read_readings("water_readings_incremental.csv"), "water_readings_incremental_staging")

In [0]:
zone_raw = spark.read.json(f"{base_path}/water_zones.json")
save_delta(zone_raw, "water_zones_raw")
zone_flat = zone_raw.select(
    "zone_id", "zone_name",
    F.col("location.city").alias("city"),
    F.col("location.district").alias("district"),
    "zone_type", "daily_capacity_liters", "active", "commissioned_on"
)
save_delta(zone_flat, "water_zones")

In [0]:
request_schema = StructType([
    StructField("request_id", IntegerType(), True),
    StructField("meter_id", StringType(), True),
    StructField("zone_id", StringType(), True),
    StructField("reported_at", TimestampType(), True),
    StructField("issue_type", StringType(), True),
    StructField("priority", StringType(), True),
    StructField("estimated_cost", DoubleType(), True),
    StructField("work_status", StringType(), True),
])
requests = (spark.read.option("header", "true")
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .schema(request_schema)
    .csv(f"{base_path}/water_service_requests.csv"))
save_delta(requests, "water_service_requests")

In [0]:
for name in [
    "water_meter_readings_source", "water_meter_readings_work",
    "water_zones_raw", "water_zones", "water_service_requests",
    "water_reading_changes", "water_readings_incremental_staging"
]:
    print(f"{catalog}.{schema}.{name}: {spark.table(f'{catalog}.{schema}.{name}').count()} rows")
print("SETUP COMPLETE. Do not rerun after beginning exercises or pipeline ingestion.")

workspace.water_monitoring.water_meter_readings_source: 157 rows
workspace.water_monitoring.water_meter_readings_work: 157 rows
workspace.water_monitoring.water_zones_raw: 11 rows
workspace.water_monitoring.water_zones: 11 rows
workspace.water_monitoring.water_service_requests: 36 rows
workspace.water_monitoring.water_reading_changes: 12 rows
workspace.water_monitoring.water_readings_incremental_staging: 18 rows
SETUP COMPLETE. Do not rerun after beginning exercises or pipeline ingestion.


**Section A | SQL Querying and Filtering**

01. Inspect water_meter_readings_source and display its schema and total row count.

In [0]:
%sql
DESCRIBE TABLE water_meter_readings_source;

col_name,data_type,comment
reading_id,int,null
meter_id,string,null
zone_id,string,null
reading_time,timestamp,null
flow_lpm,double,null
pressure_bar,double,null
chlorine_mg_l,double,null
consumption_liters,int,null
reading_status,string,null
source_system,string,null


In [0]:
%sql
SELECT COUNT(*) AS total_rows FROM water_meter_readings_source;

total_rows
157


02. Retrieve reading_id, meter_id, zone_id, reading_time and consumption_liters for readings marked Valid.

In [0]:
%sql
SELECT reading_id, meter_id, zone_id, reading_time, consumption_liters
FROM water_meter_readings_source WHERE reading_status = 'Valid';

reading_id,meter_id,zone_id,reading_time,consumption_liters
1001,M001,W01,2026-10-01T07:00:00.000Z,2365
1002,M002,W02,2026-10-01T08:10:00.000Z,3545
1003,M003,W03,2026-10-01T09:20:00.000Z,548
1004,M004,W04,2026-10-01T10:00:00.000Z,2871
1006,M006,W06,2026-10-01T12:20:00.000Z,2171
1007,M007,W07,2026-10-01T13:00:00.000Z,4167
1008,M008,W08,2026-10-01T14:10:00.000Z,3621
1009,M009,W09,2026-10-01T15:20:00.000Z,1609
1010,M010,W10,2026-10-01T16:00:00.000Z,1829
1011,M011,W01,2026-10-02T07:10:00.000Z,313


03. Find readings where pressure_bar is below 2.5, ordering the results from lowest pressure upward.

In [0]:
%sql
SELECT * FROM water_meter_readings_source 
WHERE pressure_bar < 2.5 ORDER BY pressure_bar ASC;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1071,M011,W01,2026-10-08T07:10:00.000Z,10.9,1.8,0.8,647,Valid,SmartMeter
1064,M004,W04,2026-10-07T10:00:00.000Z,20.3,1.82,0.84,708,Valid,Gateway
1024,M024,W04,2026-10-03T10:20:00.000Z,44.5,1.83,1.25,2085,Valid,FieldApp
1116,M026,W06,2026-10-12T12:10:00.000Z,28.9,1.84,0.36,1692,Valid,SmartMeter
1133,M013,W03,2026-10-14T09:00:00.000Z,15.6,1.86,0.8,900,Valid,Gateway
1028,M028,W08,2026-10-03T14:00:00.000Z,15.0,1.9,0.94,452,Valid,Gateway
1032,M002,W02,2026-10-04T08:10:00.000Z,34.4,1.92,0.88,1006,Valid,SmartMeter
1051,M021,W01,2026-10-06T07:20:00.000Z,72.0,1.93,0.27,1818,Valid,FieldApp
1052,M022,W02,2026-10-06T08:00:00.000Z,74.0,1.94,0.21,4669,Valid,Gateway
1044,M014,W04,2026-10-05T10:10:00.000Z,79.3,1.95,1.23,2661,Valid,SmartMeter


04. Find records where flow_lpm is between 20 and 50 and the source_system is Gateway or SmartMeter.

In [0]:
%sql
SELECT * FROM water_meter_readings_source
WHERE flow_lpm BETWEEN 20 AND 50 AND source_system IN ('Gateway','SmartMeter');

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway
1014,M014,W04,2026-10-02T10:10:00.000Z,22.4,4.73,0.5,1108,Valid,SmartMeter
1017,M017,W07,2026-10-02T13:10:00.000Z,28.0,3.77,1.08,1318,Valid,SmartMeter
1019,M019,W09,2026-10-02T15:00:00.000Z,23.0,4.34,0.28,1091,Valid,Gateway
1022,M022,W02,2026-10-03T08:00:00.000Z,32.5,3.35,0.6,1706,Suspect,Gateway
1032,M002,W02,2026-10-04T08:10:00.000Z,34.4,1.92,0.88,1006,Valid,SmartMeter
1047,M017,W07,2026-10-05T13:10:00.000Z,28.8,3.23,0.91,1258,Valid,SmartMeter
1049,M019,W09,2026-10-05T15:00:00.000Z,21.4,2.43,0.47,611,Valid,Gateway
1050,M020,W10,2026-10-05T16:10:00.000Z,37.2,3.91,1.02,2154,Valid,SmartMeter
1056,M026,W06,2026-10-06T12:10:00.000Z,27.2,2.69,0.68,1131,Suspect,SmartMeter


05. List distinct reading_status values and count how many readings belong to each status.

In [0]:
%sql
SELECT reading_status, COUNT(*) AS cnt FROM water_meter_readings_source GROUP BY reading_status;

reading_status,cnt
Valid,137
Suspect,11
Rejected,9


In [0]:
%sql
SELECT DISTINCT reading_status FROM water_meter_readings_source;

reading_status
Valid
Suspect
Rejected


**Section B | Joins, Aggregations and Window Functions**

06. Join meter readings with water_zones and show zone_name, district, meter_id and consumption_liters.

In [0]:
%sql
SELECT z.zone_name, z.district, r.meter_id, r.consumption_liters
FROM water_meter_readings_source r JOIN water_zones z ON r.zone_id = z.zone_id;

zone_name,district,meter_id,consumption_liters
Old City,Central,M001,2365
Hill View,North,M002,3545
Market District,Central,M003,548
Lake Side,West,M004,2871
University Quarter,East,M005,2656
Industrial Estate,South,M006,2171
Garden Colony,North,M007,4167
Transit Hub,East,M008,3621
Riverfront,West,M009,1609
South Extension,South,M010,1829


07. Find reading records whose zone_id has no matching entry in water_zones.

In [0]:
%sql
SELECT r.* FROM water_meter_readings_source r
LEFT ANTI JOIN water_zones z ON r.zone_id = z.zone_id;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1154,M999,W99,2026-10-16T11:00:00.000Z,24.5,3.0,0.45,1150,Valid,FieldApp
1156,M012,null,2026-10-16T12:30:00.000Z,28.0,3.1,0.52,870,Valid,Gateway


08. Find zones that currently have no meter readings, using a suitable join.

In [0]:
%sql
SELECT z.* FROM water_zones z
LEFT ANTI JOIN water_meter_readings_source r ON z.zone_id = r.zone_id;

zone_id,zone_name,city,district,zone_type,daily_capacity_liters,active,commissioned_on
W11,New Development,Hyderabad,West,Mixed,125000,false,2021-03-15


09. Calculate total valid consumption, average pressure and reading count for each zone; exclude nonpositive consumption.

In [0]:
%sql
SELECT zone_id, SUM(consumption_liters) total_valid_consumption,
       AVG(pressure_bar) avg_pressure, COUNT(*) reading_count
FROM water_meter_readings_source
WHERE reading_status='Valid' AND consumption_liters > 0
GROUP BY zone_id;

zone_id,total_valid_consumption,avg_pressure,reading_count
W01,23107,3.423076923076923,13
W02,24120,3.3061538461538458,13
W03,35665,3.019999999999999,13
W04,20781,3.2371428571428575,14
W06,20212,3.160769230769231,13
W07,25104,3.5700000000000003,13
W08,33689,2.8657142857142857,14
W09,20894,3.5469230769230773,13
W10,27808,3.717692307692307,14
W05,28717,3.632857142857143,14


10. Rank meters by total valid consumption within each district using a SQL window function.

In [0]:
%sql
WITH m AS (
  SELECT z.district, r.meter_id, SUM(r.consumption_liters) total
  FROM water_meter_readings_source r JOIN water_zones z USING (zone_id)
  WHERE r.reading_status='Valid' AND r.consumption_liters > 0
  GROUP BY z.district, r.meter_id)
SELECT *, RANK() OVER (PARTITION BY district ORDER BY total DESC) AS rnk FROM m;

district,meter_id,total,rnk
Central,M023,17814,1
Central,M001,11416,2
Central,M003,10447,3
Central,M013,7404,4
Central,M021,7395,5
Central,M011,4296,6
East,M008,14095,1
East,M018,10660,2
East,M025,10536,3
East,M015,9807,4


**Section C | PySpark DataFrame Operations**

In [0]:
from pyspark.sql import functions as F

11. Load the meter readings and zone master Delta tables into separate PySpark DataFrames; print their schemas.

In [0]:
readings = spark.table("water_meter_readings_source")
zones = spark.table("water_zones")
readings.printSchema(); zones.printSchema()  

root
 |-- reading_id: integer (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- reading_time: timestamp (nullable = true)
 |-- flow_lpm: double (nullable = true)
 |-- pressure_bar: double (nullable = true)
 |-- chlorine_mg_l: double (nullable = true)
 |-- consumption_liters: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_system: string (nullable = true)

root
 |-- zone_id: string (nullable = true)
 |-- zone_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- zone_type: string (nullable = true)
 |-- daily_capacity_liters: long (nullable = true)
 |-- active: boolean (nullable = true)
 |-- commissioned_on: string (nullable = true)



12. Use select(), filter() and orderBy() to display high-flow Valid readings.

In [0]:
(readings.select("reading_id","meter_id","zone_id","flow_lpm")
   .filter((F.col("reading_status")=="Valid") & (F.col("flow_lpm")>=60))
   .orderBy(F.desc("flow_lpm")).show())

+----------+--------+-------+--------+
|reading_id|meter_id|zone_id|flow_lpm|
+----------+--------+-------+--------+
|      1110|    M020|    W10|    84.0|
|      1083|    M023|    W03|    83.9|
|      1106|    M016|    W06|    83.9|
|      1010|    M010|    W10|    81.3|
|      1078|    M018|    W08|    80.7|
|      1013|    M013|    W03|    80.7|
|      1093|    M003|    W03|    80.4|
|      1031|    M001|    W01|    80.1|
|      1123|    M003|    W03|    79.7|
|      1044|    M014|    W04|    79.3|
|      1117|    M027|    W07|    79.1|
|      1026|    M026|    W06|    79.1|
|      1040|    M010|    W10|    78.9|
|      1088|    M028|    W08|    78.9|
|      1135|    M015|    W05|    78.2|
|      1077|    M017|    W07|    77.9|
|      1113|    M023|    W03|    76.3|
|      1129|    M009|    W09|    76.3|
|      1035|    M005|    W05|    76.1|
|      1136|    M016|    W06|    75.5|
+----------+--------+-------+--------+
only showing top 20 rows


13. Add a flow_category column: High for flow_lpm >= 60, Medium for 30 to below 60, Low otherwise;handle nulls explicitly.

In [0]:
readings = readings.withColumn("flow_category",
   F.when(F.col("flow_lpm").isNull(), "Unknown")
    .when(F.col("flow_lpm")>=60, "High")
    .when(F.col("flow_lpm")>=30, "Medium")
    .otherwise("Low"))     
display(readings)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system,flow_category
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway,Medium
1002,M002,W02,2026-10-01T08:10:00.000Z,59.9,4.6,0.39,3545,Valid,SmartMeter,Medium
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp,Low
1004,M004,W04,2026-10-01T10:00:00.000Z,56.7,2.84,1.1,2871,Valid,Gateway,Medium
1005,M005,W05,2026-10-01T11:10:00.000Z,61.8,3.04,0.43,2656,Suspect,SmartMeter,High
1006,M006,W06,2026-10-01T12:20:00.000Z,35.8,2.52,1.2,2171,Valid,FieldApp,Medium
1007,M007,W07,2026-10-01T13:00:00.000Z,65.0,3.19,0.67,4167,Valid,Gateway,High
1008,M008,W08,2026-10-01T14:10:00.000Z,70.3,3.39,0.54,3621,Valid,SmartMeter,High
1009,M009,W09,2026-10-01T15:20:00.000Z,30.8,4.57,1.05,1609,Valid,FieldApp,Medium
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway,High


14. Join the DataFrames and calculate average flow_lpm for each zone_type.

In [0]:
(readings.join(zones, "zone_id").groupBy("zone_type")
   .agg(F.avg("flow_lpm").alias("avg_flow")).show()) 

+-------------+------------------+
|    zone_type|          avg_flow|
+-------------+------------------+
|   Commercial| 56.10000000000001|
|  Residential|45.155737704918025|
|        Mixed|         38.165625|
|   Industrial| 43.14374999999999|
|Institutional|          49.33125|
+-------------+------------------+



15. Use PySpark groupBy() and agg() to identify the three zones with the highest valid water consumption.

In [0]:
(readings.filter((F.col("reading_status")=="Valid") & (F.col("consumption_liters")>0))
   .groupBy("zone_id").agg(F.sum("consumption_liters").alias("total"))
   .orderBy(F.desc("total")).limit(3).show()) 

+-------+-----+
|zone_id|total|
+-------+-----+
|    W03|35665|
|    W08|33689|
|    W05|28717|
+-------+-----+



**Section D | Data Quality and Cleaning**

16. Identify duplicate reading_id values and display every row associated with those IDs.

In [0]:
%sql
SELECT * FROM water_meter_readings_source
WHERE reading_id IN (SELECT reading_id FROM water_meter_readings_source
                     GROUP BY reading_id HAVING COUNT(*)>1)
ORDER BY reading_id;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1044,M014,W04,2026-10-05T10:10:00.000Z,79.3,1.95,1.23,2661,Valid,SmartMeter
1044,M014,W04,2026-10-16T13:00:00.000Z,36.0,3.2,0.8,1300,Valid,Gateway


17. Find missing pressure_bar and source_system values and count the affected records.

In [0]:
%sql
SELECT SUM(CASE WHEN pressure_bar IS NULL THEN 1 ELSE 0 END) missing_pressure,
       SUM(CASE WHEN source_system IS NULL THEN 1 ELSE 0 END) missing_source
FROM water_meter_readings_source;

missing_pressure,missing_source
1,1


18. Identify invalid measurements: negative flow_lpm, nonpositive consumption_liters or missing zone_id.

In [0]:
%sql
SELECT * FROM water_meter_readings_source
WHERE flow_lpm < 0 OR consumption_liters <= 0 OR zone_id IS NULL;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1151,M004,W04,2026-10-16T08:00:00.000Z,-12.5,2.8,0.62,500,Suspect,Gateway
1152,M015,W05,2026-10-16T09:00:00.000Z,22.0,2.1,0.5,0,Valid,SmartMeter
1156,M012,null,2026-10-16T12:30:00.000Z,28.0,3.1,0.52,870,Valid,Gateway


19. Create water_readings_clean as a new Delta table containing one row per reading_id, with Valid status and positive measurements; document the duplicate-selection rule.

In [0]:
%sql
CREATE OR REPLACE TABLE water_readings_clean AS
SELECT * EXCEPT (rn) FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY reading_id
            ORDER BY reading_time DESC, source_system) rn
  FROM water_meter_readings_source
  WHERE reading_status='Valid' AND flow_lpm>0 AND consumption_liters>0
        AND pressure_bar>0)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


20. Use the service request data to identify zones with High-priority requests and compare their average recorded pressure.

In [0]:
%sql
WITH hp AS (SELECT DISTINCT zone_id FROM water_service_requests WHERE priority='High')
SELECT CASE WHEN hp.zone_id IS NOT NULL THEN 'High-priority zone' ELSE 'Other' END grp,
       AVG(r.pressure_bar) avg_pressure
FROM water_readings_clean r LEFT JOIN hp USING (zone_id) GROUP BY 1;

grp,avg_pressure
High-priority zone,3.3562878787878785
Other,3.05


**Section E | Delta Lake Tables and CRUD**

21. Create water_readings_work_copy using CREATE TABLE AS SELECT from water_meter_readings_work.

In [0]:
%sql
CREATE OR REPLACE TABLE water_readings_work_copy AS SELECT * FROM water_meter_readings_work; 

num_affected_rows,num_inserted_rows


22. Insert two new readings into water_readings_work_copy using SQL; use new unique reading IDs.

In [0]:
%sql
INSERT INTO water_readings_work_copy (reading_id, meter_id, zone_id, reading_time,
  flow_lpm, pressure_bar, consumption_liters, source_system, reading_status)
VALUES (9001,'M900','W01',current_timestamp(),35.0,3.1,420.0,'Gateway','Valid'),
       (9002,'M901','W02',current_timestamp(),42.0,3.4,510.0,'SmartMeter','Valid');

num_affected_rows,num_inserted_rows
2,2


23. Update the status of a chosen Suspect reading in the copy to Rejected and verify the change.

In [0]:
%sql
SELECT reading_id FROM water_readings_work_copy WHERE reading_status='Suspect' LIMIT 1;

reading_id
1005


In [0]:
%sql
UPDATE water_readings_work_copy SET reading_status='Rejected' WHERE reading_id = 1005;

num_affected_rows
1


In [0]:
%sql
SELECT * FROM water_readings_work_copy WHERE reading_id = 1005;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1005,M005,W05,2026-10-01T11:10:00.000Z,61.8,3.04,0.43,2656,Rejected,SmartMeter


24. Delete Rejected readings from the copy and compare counts before and after deletion.

In [0]:
%sql
SELECT COUNT(*) FROM water_readings_work_copy;

COUNT(*)
159


In [0]:
%sql
DELETE FROM water_readings_work_copy WHERE reading_status='Rejected';

num_affected_rows
10


In [0]:
%sql
SELECT COUNT(*) FROM water_readings_work_copy;

COUNT(*)
149


25. Use PySpark to create a Delta table named zone_consumption_summary from an aggregated DataFrame.

In [0]:
(spark.table("water_readings_clean").groupBy("zone_id")
  .agg(F.sum("consumption_liters").alias("total_consumption"),
       F.avg("pressure_bar").alias("avg_pressure"), F.count("*").alias("n"))
  .write.format("delta").mode("overwrite").saveAsTable("zone_consumption_summary"))

**Section F | MERGE, UPSERT and SCD**

26. Compare water_reading_changes with water_meter_readings_work and separate matching IDs from new IDs.

In [0]:
%sql
SELECT c.* FROM water_reading_changes c JOIN water_meter_readings_work w USING (reading_id); 

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1005,M005,W05,2026-10-01T11:10:00.000Z,64.3,3.24,0.43,2776,Valid,FieldApp
1027,M027,W07,2026-10-03T13:20:00.000Z,54.6,4.99,0.95,2678,Valid,FieldApp
1062,M002,W02,2026-10-07T08:10:00.000Z,30.1,2.43,0.3,1057,Valid,FieldApp
1088,M028,W08,2026-10-09T14:00:00.000Z,81.4,2.17,1.06,4931,Valid,FieldApp
1119,M029,W09,2026-10-12T15:10:00.000Z,22.1,3.63,1.13,1339,Valid,FieldApp
1145,M025,W05,2026-10-15T11:00:00.000Z,77.5,3.96,1.29,4908,Valid,FieldApp


In [0]:
%sql
SELECT c.* FROM water_reading_changes c LEFT ANTI JOIN water_meter_readings_work w USING (reading_id);

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1201,M002,W02,2026-10-18T08:00:00.000Z,24.0,2.6,0.45,1100,Valid,FieldApp
1202,M005,W05,2026-10-18T09:00:00.000Z,28.7,2.7,0.5,1250,Valid,FieldApp
1203,M008,W08,2026-10-18T10:00:00.000Z,33.4,2.8,0.55,1400,Valid,FieldApp
1204,M011,W01,2026-10-18T11:00:00.000Z,38.1,2.9,0.6,1550,Valid,FieldApp
1205,M014,W04,2026-10-18T12:00:00.000Z,42.8,3.0,0.65,1700,Valid,FieldApp
1206,M017,W07,2026-10-18T13:00:00.000Z,47.5,3.1,0.7,1850,Valid,FieldApp


27. Run a SQL MERGE on water_meter_readings_work using reading_id as the key: update matches and insert nonmatches.

In [0]:
%sql
MERGE INTO water_meter_readings_work t
USING water_reading_changes s ON t.reading_id = s.reading_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
12,6,0,6


28. Validate that changed reading IDs have new values and that new IDs were inserted.

In [0]:
%sql
SELECT s.reading_id, s.flow_lpm AS new_flow, t.flow_lpm AS table_flow
FROM water_reading_changes s JOIN water_meter_readings_work t USING (reading_id);

reading_id,new_flow,table_flow
1005,64.3,64.3
1027,54.6,54.6
1062,30.1,30.1
1088,81.4,81.4
1119,22.1,22.1
1145,77.5,77.5
1201,24.0,24.0
1202,28.7,28.7
1203,33.4,33.4
1204,38.1,38.1


29. Perform the same SCD Type 1 MERGE using the PySpark DeltaTable API on a fresh working-table copy.

In [0]:
%sql
DESCRIBE HISTORY water_meter_readings_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-11T08:43:02.000Z,72240446752486,tsaravanapriya535@gmail.com,MERGE,"Map(predicate -> [""(reading_id#19406 = reading_id#19416)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1421029179937148),119803cb-fd6c-49dc-8ef7-e6a8e59d2cdd,1111-083843-66bbfrib-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4170, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 6, executionTimeMs -> 3186, materializeSourceTimeMs -> 7, numTargetRowsInserted -> 6, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1070, numTargetRowsUpdated -> 6, numOutputRows -> 12, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 12, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2034)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-11T08:39:54.000Z,72240446752486,tsaravanapriya535@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""dfab3f80-b3bb-4082-82a1-da138db70c7f"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-6bb61d84-99bf-48ab-992e-5c42f620e525"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-7add51d8-e9e3-410c-8e75-656e22a788ac""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1421029179937148),94971644-b13c-4c74-b4b6-762b74a98c95,1111-083843-66bbfrib-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 5424, numDeletionVectorsRemoved -> 0, numOutputRows -> 157, numOutputBytes -> 5424)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-11T07:24:00.000Z,72240446752486,tsaravanapriya535@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""dfab3f80-b3bb-4082-82a1-da138db70c7f"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-6bb61d84-99bf-48ab-992e-5c42f620e525"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-7add51d8-e9e3-410c-8e75-656e22a788ac""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1421029179937148),5f5dd540-9067-4ee5-8d4c-1d05f63dea25,1111-041805-vwancx9k-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 157, numOutp

In [0]:
%sql
CREATE OR REPLACE TABLE water_work_pyspark AS
SELECT * FROM water_meter_readings_work VERSION AS OF 1;

num_affected_rows,num_inserted_rows


In [0]:
from delta.tables import DeltaTable
dt = DeltaTable.forName(spark, "water_work_pyspark")
(dt.alias("t").merge(spark.table("water_reading_changes").alias("s"), "t.reading_id = s.reading_id")
   .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

30. Build a zone_capacity_history SCD Type 2 table from water_zones. Effective 2026-10-20, change W02 capacity to 195000 and W06 to 305000, retaining old versions and current flags.

In [0]:
%sql
CREATE OR REPLACE TABLE zone_capacity_history AS
SELECT zone_id, zone_name, daily_capacity_liters, DATE'2000-01-01' AS effective_from,
        DATE'9999-12-31' AS effective_to, true AS is_current
FROM water_zones;   

CREATE OR REPLACE TEMP VIEW cap_changes AS
SELECT * FROM VALUES ('W02',195000),('W06',305000) AS v(zone_id,new_capacity);

-- 1) insert new versions (before closing old rows)
INSERT INTO zone_capacity_history
SELECT h.zone_id, h.zone_name, c.new_capacity, DATE'2026-10-20', DATE'9999-12-31', true
FROM zone_capacity_history h JOIN cap_changes c ON h.zone_id=c.zone_id
WHERE h.is_current = true;
-- 2) close old versions
UPDATE zone_capacity_history SET is_current=false, effective_to=DATE'2026-10-20'
WHERE is_current = true AND effective_from < DATE'2026-10-20'
AND zone_id IN ('W02','W06');

num_affected_rows
2


**Section G | Delta History, Time Travel and Maintenance**

31. Use DESCRIBE DETAIL on water_meter_readings_work to inspect the format, storage location, file count and size.

In [0]:
%sql
DESCRIBE DETAIL water_meter_readings_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,8bcbcf61-169a-4e89-8e6c-18029e15789d,workspace.water_monitoring.water_meter_readings_work,null,,2026-11-11T07:23:58.410Z,2026-11-11T08:43:02.000Z,List(),List(),2,9594,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> dfab3f80-b3bb-4082-82a1-da138db70c7f, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-6bb61d84-99bf-48ab-992e-5c42f620e525, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-7add51d8-e9e3-410c-8e75-656e22a788ac)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 6, numDeletionVectors -> 1)",true


32. Use DESCRIBE HISTORY to identify recorded table operations and locate the version produced by MERGE.

In [0]:
%sql
DESCRIBE HISTORY water_meter_readings_work; 

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-11T08:43:02.000Z,72240446752486,tsaravanapriya535@gmail.com,MERGE,"Map(predicate -> [""(reading_id#19406 = reading_id#19416)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1421029179937148),119803cb-fd6c-49dc-8ef7-e6a8e59d2cdd,1111-083843-66bbfrib-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4170, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 6, executionTimeMs -> 3186, materializeSourceTimeMs -> 7, numTargetRowsInserted -> 6, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1070, numTargetRowsUpdated -> 6, numOutputRows -> 12, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 12, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2034)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-11T08:39:54.000Z,72240446752486,tsaravanapriya535@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""dfab3f80-b3bb-4082-82a1-da138db70c7f"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-6bb61d84-99bf-48ab-992e-5c42f620e525"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-7add51d8-e9e3-410c-8e75-656e22a788ac""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1421029179937148),94971644-b13c-4c74-b4b6-762b74a98c95,1111-083843-66bbfrib-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 5424, numDeletionVectorsRemoved -> 0, numOutputRows -> 157, numOutputBytes -> 5424)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-11T07:24:00.000Z,72240446752486,tsaravanapriya535@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""dfab3f80-b3bb-4082-82a1-da138db70c7f"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-6bb61d84-99bf-48ab-992e-5c42f620e525"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-7add51d8-e9e3-410c-8e75-656e22a788ac""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1421029179937148),5f5dd540-9067-4ee5-8d4c-1d05f63dea25,1111-041805-vwancx9k-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 157, numOutp

33. Query the version immediately before the MERGE using VERSION AS OF; compare reading 1005 with its current value.

In [0]:
%sql
SELECT * FROM water_meter_readings_work VERSION AS OF 1 WHERE reading_id=1005;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1005,M005,W05,2026-10-01T11:10:00.000Z,61.8,3.04,0.43,2656,Suspect,SmartMeter


In [0]:
%sql
SELECT * FROM water_meter_readings_work WHERE reading_id=1005;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1005,M005,W05,2026-10-01T11:10:00.000Z,64.3,3.24,0.43,2776,Valid,FieldApp


34. Execute OPTIMIZE on a suitable working Delta table; compare its file count before and after and explain the result.

In [0]:
%sql
DESCRIBE DETAIL water_readings_work_copy; 

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,0761310a-8029-49c1-9ff1-a385fcc0d5c7,workspace.water_monitoring.water_readings_work_copy,null,,2026-11-11T08:40:57.018Z,2026-11-11T08:41:15.000Z,List(),List(),2,8288,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> a95b93ef-517d-4a8f-a267-f81d7020e910, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-8cd9cd63-ed6f-4520-aa40-5a6a094a5b15, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-8e3ef6f6-3705-4ec5-8671-521bf0fec63c)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 10, numDeletionVectors -> 1)",true


In [0]:
%sql
OPTIMIZE water_readings_work_copy;

path,metrics
,"List(1, 2, List(6407, 6407, 6407.0, 1, 6407), List(2864, 5424, 4144.0, 2, 8288), 0, null, null, 0, 1, 2, 0, true, 0, 0, 1794387488061, 1794387490585, 8, 1, null, List(1, 10), null, 10, 10, 543, 0, null, null, 0)"


In [0]:
%sql
DESCRIBE DETAIL water_readings_work_copy; 

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,0761310a-8029-49c1-9ff1-a385fcc0d5c7,workspace.water_monitoring.water_readings_work_copy,null,,2026-11-11T08:40:57.018Z,2026-11-11T08:58:11.000Z,List(),List(),1,6407,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> a95b93ef-517d-4a8f-a267-f81d7020e910, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-8cd9cd63-ed6f-4520-aa40-5a6a094a5b15, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-8e3ef6f6-3705-4ec5-8671-521bf0fec63c)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


OPTIMIZE: small tables may show little change, such as 1 file to 1 file. Say so and explain that compaction only matters with many small files.

35. Explain the purpose of VACUUM and inspect eligible files with a dry run where supported; do not delete retained historical files.

In [0]:
%sql
VACUUM water_readings_work_copy DRY RUN;

path


VACUUM: it deletes data files no longer referenced by the table and older than the retention period (7 days by default). That is why it can break time travel to old versions.

**Section H | Unity Catalog, Views and Parquet**

36. Identify the catalog and schema containing your assessment tables; display the managed Delta table location.

In [0]:
%sql
SELECT current_catalog(), current_schema(); 

current_catalog(),current_schema()
workspace,water_monitoring


37. Create a permanent view for valid readings with zone names and districts.

In [0]:
%sql
CREATE OR REPLACE VIEW valid_readings_vw AS                           
SELECT r.*, z.zone_name, z.district FROM water_meter_readings_source r
JOIN water_zones z USING (zone_id) WHERE r.reading_status='Valid';

In [0]:
%sql
SELECT * FROM valid_readings_vw

zone_id,reading_id,meter_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system,zone_name,district
W01,1001,M001,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway,Old City,Central
W02,1002,M002,2026-10-01T08:10:00.000Z,59.9,4.6,0.39,3545,Valid,SmartMeter,Hill View,North
W03,1003,M003,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp,Market District,Central
W04,1004,M004,2026-10-01T10:00:00.000Z,56.7,2.84,1.1,2871,Valid,Gateway,Lake Side,West
W06,1006,M006,2026-10-01T12:20:00.000Z,35.8,2.52,1.2,2171,Valid,FieldApp,Industrial Estate,South
W07,1007,M007,2026-10-01T13:00:00.000Z,65.0,3.19,0.67,4167,Valid,Gateway,Garden Colony,North
W08,1008,M008,2026-10-01T14:10:00.000Z,70.3,3.39,0.54,3621,Valid,SmartMeter,Transit Hub,East
W09,1009,M009,2026-10-01T15:20:00.000Z,30.8,4.57,1.05,1609,Valid,FieldApp,Riverfront,West
W10,1010,M010,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway,South Extension,South
W01,1011,M011,2026-10-02T07:10:00.000Z,13.0,4.39,0.66,313,Valid,SmartMeter,Old City,Central


38. Create a session-scoped temporary view for low-pressure readings and query it.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW low_pressure_tv AS
SELECT * FROM water_meter_readings_source WHERE pressure_bar < 2.5;

In [0]:
%sql
SELECT * FROM low_pressure_tv;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway
1016,M016,W06,2026-10-02T12:00:00.000Z,12.9,2.26,0.72,613,Valid,Gateway
1018,M018,W08,2026-10-02T14:20:00.000Z,70.5,2.09,0.3,3709,Valid,FieldApp
1024,M024,W04,2026-10-03T10:20:00.000Z,44.5,1.83,1.25,2085,Valid,FieldApp
1028,M028,W08,2026-10-03T14:00:00.000Z,15.0,1.9,0.94,452,Valid,Gateway
1031,M001,W01,2026-10-04T07:00:00.000Z,80.1,2.38,1.21,3504,Valid,Gateway
1032,M002,W02,2026-10-04T08:10:00.000Z,34.4,1.92,0.88,1006,Valid,SmartMeter
1033,M003,W03,2026-10-04T09:20:00.000Z,56.6,2.19,1.24,2184,Valid,FieldApp
1036,M006,W06,2026-10-04T12:20:00.000Z,26.2,2.09,0.56,1065,Valid,FieldApp


39. Create and query a global temporary view of High-priority service requests if your compute supports it; explain its scope.

In [0]:
%sql
CREATE OR REPLACE GLOBAL TEMP VIEW high_priority_req AS
SELECT * FROM water_service_requests WHERE priority='High';

40. Write a DataFrame to Parquet in an authorized Unity Catalog Volume, then load that Parquet dataset into a new managed Delta table.

In [0]:
p = "/Volumes/workspace/water_monitoring/landing/parquet_output/readings"      
spark.table("water_readings_clean").write.mode("overwrite").parquet(p)
spark.read.parquet(p).write.format("delta").mode("overwrite").saveAsTable("water_readings_from_parquet")

**Section I | Structured Streaming**

41. Create a streaming DataFrame using the built-in rate source with five rows per second.

In [0]:
stream = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

42. Transform the generated value into a simulated meter_id and flow_lpm column.

In [0]:
sim = (stream.withColumn("meter_id", F.concat(F.lit("M"), (F.col("value")%20).cast("string")))
             .withColumn("flow_lpm", (F.col("value")%80).cast("double")))

43. Filter the stream to retain simulated flow_lpm above a threshold of your choice.

In [0]:
high = sim.filter(F.col("flow_lpm") > 40)

44. Aggregate streaming records by a derived flow category and display the live counts using a supported sink.

In [0]:
agg = (high.withColumn("flow_category", F.when(F.col("flow_lpm")>=60,"High").otherwise("Medium"))
           .groupBy("flow_category").count())

45. Stop the running streaming query and explain the role of a checkpoint for durable streaming output.

In [0]:
ckpt = "/Volumes/workspace/water_monitoring/landing/checkpoints/rate_q44"
dbutils.fs.rm(ckpt, True)

q = (agg.writeStream
     .format("delta")
     .outputMode("complete")
     .option("checkpointLocation", ckpt)
     .trigger(availableNow=True)      
     .toTable("rate_flow_category_counts"))

q.awaitTermination()                  
display(spark.table("rate_flow_category_counts"))

flow_category,count


**Section J | Lakeflow / Delta Live Tables**

46. Create a SQL Lakeflow pipeline with a Bronze streaming table that reads append-only water_meter_readings_source.

47. Create a Silver streaming table filtering Valid readings with positive flow and consumption.

48. Create a Gold materialized view aggregating total consumption, average pressure and reading counts by zone.

49. Run the SQL pipeline; inspect its dependency graph and verify Bronze, Silver and Gold outputs.

50. Create a separate Python Lakeflow pipeline using pyspark.pipelines to implement equivalent Bronze, Silver and Gold tables with different names.

For solutions of this section, kindly refer the Section_J_Pipelines Folder and Screenshots folder in Azure Databricks_Assessment_02.

**Section K | Incremental Loads, Analytics and Visualizations**

51. Inspect water_readings_incremental_staging and append its records to water_meter_readings_source without overwriting existing source data.

In [0]:
%sql
SELECT * FROM water_readings_incremental_staging;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1161,M003,W03,2026-10-19T07:00:00.000Z,17.0,2.1,0.35,700,Suspect,Gateway
1162,M005,W05,2026-10-19T08:00:00.000Z,19.3,2.38,0.48,790,Valid,SmartMeter
1163,M007,W07,2026-10-19T09:00:00.000Z,21.6,2.66,0.61,880,Valid,Gateway
1164,M009,W09,2026-10-19T10:00:00.000Z,23.9,2.94,0.74,970,Valid,SmartMeter
1165,M011,W01,2026-10-19T11:00:00.000Z,26.2,3.22,0.87,1060,Valid,Gateway
1166,M013,W03,2026-10-19T12:00:00.000Z,28.5,3.5,0.35,1150,Valid,SmartMeter
1167,M015,W05,2026-10-20T07:00:00.000Z,30.8,2.1,0.48,1240,Valid,Gateway
1168,M017,W07,2026-10-20T08:00:00.000Z,33.1,2.38,0.61,1330,Suspect,SmartMeter
1169,M019,W09,2026-10-20T09:00:00.000Z,35.4,2.66,0.74,1420,Valid,Gateway
1170,M021,W01,2026-10-20T10:00:00.000Z,37.7,2.94,0.87,1510,Valid,SmartMeter


In [0]:
%sql
INSERT INTO water_meter_readings_source SELECT * FROM water_readings_incremental_staging;

num_affected_rows,num_inserted_rows
18,18


52. Rerun the SQL Lakeflow pipeline and verify that the Gold results reflect the newly appended readings.

For solution, Kindly refer Q52 in Section_J_Pipelines in Azure Databricks_Assessment_02.

53. Build a bar chart showing total valid consumption by zone and a pie chart showing consumption share by district.

In [0]:
%sql
SELECT COALESCE(z.zone_name, s.zone_id, 'Unknown') AS zone,
       SUM(s.consumption_liters) AS total_consumption
FROM workspace.water_monitoring.water_silver s
LEFT JOIN workspace.water_monitoring.water_zones z ON s.zone_id = z.zone_id
GROUP BY 1
ORDER BY total_consumption DESC;

zone,total_consumption
Market District,40465
University Quarter,34577
Transit Hub,33689
Garden Colony,29994
South Extension,27808
Old City,25677
Riverfront,25154
Hill View,24120
Lake Side,20781
Industrial Estate,20212


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT z.district, SUM(s.consumption_liters) AS total_consumption
FROM workspace.water_monitoring.water_silver s
JOIN workspace.water_monitoring.water_zones z ON s.zone_id = z.zone_id
GROUP BY z.district;

district,total_consumption
North,54114
East,68266
South,48020
Central,66142
West,45935


Databricks visualization. Run in Databricks to view.

54. Build a line chart showing daily total consumption and a scatter plot of pressure_bar versus flow_lpm for valid readings.

In [0]:
%sql
SELECT CAST(reading_time AS DATE) AS reading_date,
       SUM(consumption_liters) AS daily_consumption
FROM workspace.water_monitoring.water_silver
GROUP BY 1
ORDER BY 1;

reading_date,daily_consumption
2026-10-01,22726
2026-10-02,13169
2026-10-03,14187
2026-10-04,19704
2026-10-05,16271
2026-10-06,19793
2026-10-07,11012
2026-10-08,14937
2026-10-09,19235
2026-10-10,16787


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT pressure_bar, flow_lpm, zone_id
FROM workspace.water_monitoring.water_silver
WHERE pressure_bar IS NOT NULL AND flow_lpm IS NOT NULL;

pressure_bar,flow_lpm,zone_id
2.38,19.3,W05
2.66,21.6,W07
2.94,23.9,W09
3.22,26.2,W01
3.5,28.5,W03
2.1,30.8,W05
2.66,35.4,W09
2.94,37.7,W01
3.22,40.0,W03
3.5,42.3,W05


Databricks visualization. Run in Databricks to view.

55. Create a final dashboard or notebook report combining zone consumption, water-quality indicators and service request counts; explain two operational findings.

**Dataset 1, ds_zone_summary: consumption, pressure and requests per zone**

In [0]:
%sql
SELECT g.zone_id, g.total_consumption, ROUND(g.avg_pressure, 2) AS avg_pressure,
       g.reading_count, COALESCE(r.requests, 0) AS service_requests,
       COALESCE(r.high_priority, 0) AS high_priority_requests
FROM workspace.water_monitoring.water_gold_zone g
LEFT JOIN (SELECT zone_id, COUNT(*) AS requests,
                  COUNT_IF(priority = 'High') AS high_priority
           FROM workspace.water_monitoring.water_service_requests
           GROUP BY zone_id) r USING (zone_id)
ORDER BY g.total_consumption DESC;

zone_id,total_consumption,avg_pressure,reading_count,service_requests,high_priority_requests
W03,40465,3.06,16,3,1
W05,34577,3.45,18,4,1
W08,33689,2.87,14,4,1
W07,29994,3.42,16,3,1
W10,27808,3.72,14,3,1
W01,25677,3.38,15,4,2
W09,25154,3.38,16,4,1
W02,24120,3.31,13,4,2
W04,20781,3.24,14,3,1
W06,20212,3.16,13,4,1


Databricks visualization. Run in Databricks to view.

**Dataset 2, ds_quality: water-quality indicators**

In [0]:
%sql
SELECT 'Low pressure (<2.5 bar)' AS indicator, COUNT_IF(pressure_bar < 2.5) AS readings
FROM workspace.water_monitoring.water_meter_readings_source
UNION ALL
SELECT 'Missing pressure', COUNT_IF(pressure_bar IS NULL)
FROM workspace.water_monitoring.water_meter_readings_source
UNION ALL
SELECT 'Not Valid status', COUNT_IF(reading_status <> 'Valid')
FROM workspace.water_monitoring.water_meter_readings_source
UNION ALL
SELECT 'Unknown / missing zone', COUNT(*)
FROM workspace.water_monitoring.water_meter_readings_source r
LEFT ANTI JOIN workspace.water_monitoring.water_zones z ON r.zone_id = z.zone_id;

indicator,readings
Low pressure (<2.5 bar),39
Missing pressure,1
Not Valid status,23
Unknown / missing zone,2


Databricks visualization. Run in Databricks to view.

**Dataset 3, ds_requests: service request counts**

In [0]:
%sql
SELECT zone_id, priority, COUNT(*) AS request_count
FROM workspace.water_monitoring.water_service_requests
GROUP BY zone_id, priority;

zone_id,priority,request_count
W01,High,2
W08,Medium,2
W05,Low,2
W02,High,2
W09,Medium,2
W06,Low,2
W03,High,1
W10,Medium,1
W07,Low,1
W04,High,1


Databricks visualization. Run in Databricks to view.

**Two operational findings**

**1.Pressure risk in W08.** W08 had the lowest average pressure, about 2.87 bar, which is close to the 2.5 bar low-pressure threshold. Low pressure alongside high consumption (33,689 L in the first run) can indicate a leak or a supply restriction. If W08 also has High-priority service requests in the table, inspect it first.

**2.Highest demand in W03, plus a data-quality gap.** W03 had the highest consumption in the first run (35,665 L), so it is the first place to check for leakage losses and capacity planning. Separately, readings from the unknown zone W99 and the missing-zone rows pass the Silver filter. Recommend a zone-validity check at ingestion so unassigned readings don’t distort zone totals.

**Final capstone | Water Operations Analytics**

• Build a Bronze-Silver-Gold Lakeflow solution from the supplied meter readings and zone master.

• Publish clean readings and per-zone and per-district summaries; flag potential low-pressure and water-quality concerns.

• Demonstrate one MERGE, one historical Delta query, an incremental pipeline update, and four visualizations.

• Submit your notebooks, pipeline definitions, pipeline graph screenshot, and chart screenshots.

For solutions, kindly refer the Capstone_Pipeline_02 in Azure Databricks_Assessment_02.